**Author:** Amanda Baright

**Date:** 04.21.2026

**Purpose:** ST 554 Homework 10

You are tasked with using spark structured streaming to deal with streaming data!

# Part 1: Creating Streaming Data Using `rate`

We first want to setup a data stream using the "rate" formula from class.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
rateDF = spark.readStream.format("rate").load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 19:58:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 19:58:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
rateDF # look at the rateDF object

DataFrame[timestamp: timestamp, value: bigint]

We then want to set up a sequence of actions using appropriate functions from `pyspark.sql.functions` that uses the `rate` data to find the square root of the rate 'value' and find mod 4 of the rate 'value'. Here we will use the `pyspark.sql.functions` of `col`, `sqrt`, and `expr`.

In [4]:
from pyspark.sql.functions import col, sqrt, expr

transformedDF = rateDF.select(
    col("timestamp"), # grab the 'timestamp' column
    col("value"), # grab the 'value' column
    # Find the square root of the rate 'value' 
    sqrt(col("value")).alias("sqrt_value"),
    # Find mod 4 of the rate 'value' 
    # We can use expr() for SQL-like syntax for the modulo operator
    expr("value % 4").alias("mod_4_value")
)

We now want to output this by creating a `writeStream` that writes to 'memory' (`format("memory")`). We will give the query a name (`queryName("...")`) and start it. We will let it run for about 30 seconds and then we will stop the query.

In [6]:
query = transformedDF.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("homework_table") \
    .start()

26/04/21 20:10:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e45273af-487f-43d5-a079-015bc4435527. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 20:10:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Here we will use the `time` function to `sleep` for 30 seconds and then run `query.stop()` to stope the query.

In [7]:
import time
time.sleep(30)
query.stop()

26/04/21 20:10:55 WARN DAGScheduler: Failed to cancel job group b1c4f319-bccb-4851-ad99-bb13d8a9a8d7. Cannot find active jobs for it.
26/04/21 20:10:55 WARN DAGScheduler: Failed to cancel job group b1c4f319-bccb-4851-ad99-bb13d8a9a8d7. Cannot find active jobs for it.


We then want to output the entire table stored int he query name using `spark.sql("select * from homework_table").show()`.

In [8]:
spark.sql("SELECT * FROM homework_table").show()

+--------------------+-----+------------------+-----------+
|           timestamp|value|        sqrt_value|mod_4_value|
+--------------------+-----+------------------+-----------+
|2026-04-21 20:10:...|    0|               0.0|          0|
|2026-04-21 20:10:...|    1|               1.0|          1|
|2026-04-21 20:10:...|    2|1.4142135623730951|          2|
|2026-04-21 20:10:...|    3|1.7320508075688772|          3|
|2026-04-21 20:10:...|    4|               2.0|          0|
|2026-04-21 20:10:...|    5|  2.23606797749979|          1|
|2026-04-21 20:10:...|    6| 2.449489742783178|          2|
|2026-04-21 20:10:...|    7|2.6457513110645907|          3|
|2026-04-21 20:10:...|    8|2.8284271247461903|          0|
|2026-04-21 20:10:...|    9|               3.0|          1|
|2026-04-21 20:10:...|   10|3.1622776601683795|          2|
|2026-04-21 20:10:...|   11|   3.3166247903554|          3|
|2026-04-21 20:10:...|   12|3.4641016151377544|          0|
|2026-04-21 20:10:...|   13| 3.605551275